# Mental Health Lifestyle Model
Educational comparison of corrected single-neuron training, logistic regression, a hidden-layer network, and a baseline. See README.md for setup and limitations.

In [ ]:
# FIXED: explicit labels, correct BCE gradients, scaled inputs, stratified split,
# matching logistic objective, baseline, class-sensitive metrics, hidden-layer model.
# Original model is a single sigmoid neuron (mathematically logistic regression).
from pathlib import Path
if not Path('/content/drive/My Drive/Collab Docs/Mental_Health_Lifestyle_Dataset.csv').is_file():
    from google.colab import drive
    drive.mount('/content/drive')

# learn about collinerity and the droping of the catgaory
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import linear_model
from sklearn.metrics import confusion_matrix

np.random.seed(42)

# Read the data
df = pd.read_csv('/content/drive/My Drive/Collab Docs/Mental_Health_Lifestyle_Dataset.csv', keep_default_na=False)

# Keep literal 'None' distinct from missing/unknown labels.
df['condition_2'] = df['Mental Health Condition'].str.strip().str.lower()
label_map = {'none': 0, 'anxiety': 1, 'depression': 1, 'bipolar': 1, 'ptsd': 1}
unknown = set(df['condition_2']) - set(label_map)
if unknown:
    raise ValueError(f'Missing or unexpected target labels: {unknown}')

# Explicit label mapping: None=0, a recorded condition=1.
df['condition_binary'] = df['condition_2'].map(label_map)

# Prepare features
X_continuous = df[['Happiness Score']].values

# Z
# Get unique categories
unique_genders = np.array(['Male', 'Female', 'Other'])
unique_diets = np.array(['Vegetarian', 'Vegan', 'Balanced', 'Junk Food', 'Keto'])
for column, allowed in [('Gender', unique_genders), ('Diet Type', unique_diets)]:
    if not df[column].isin(allowed).all():
        raise ValueError(f'Unknown or missing category in {column}')

print(f"Genders: {unique_genders}")
print(f"Diet Types: {unique_diets}")

# Makes Gender array so only one can be 1, rest are 0
gender_cols = []
for gender in unique_genders:
    col_name = f'gender_{gender}'
    df[col_name] = (df['Gender'] == gender).astype(int)
    gender_cols.append(col_name)

# Makes Diet Type array so only one can be 1, rest are 0
diet_cols = []
for diet in unique_diets:
    col_name = f'diet_{diet}'
    df[col_name] = (df['Diet Type'] == diet).astype(int)
    diet_cols.append(col_name)

# Combine all Z features and drops the last catagory because it is unnecessary
z_features = gender_cols[:-1] + diet_cols[:-1]

# Gets values of those columns
Z_categorical = df[z_features].values

# Combining X and Z
X_combined = np.concatenate([X_continuous, Z_categorical], axis=1)

# Target variable
y = df['condition_binary'].values.reshape(-1, 1)

print(f"\nTarget distribution: {np.mean(y):.3f} positive cases")

# Stratify the split and fit the happiness scaler on training rows only.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2,
                                     random_state=42, stratify=y.ravel())
x_train, x_test = X_combined[train_idx].astype(float), X_combined[test_idx].astype(float)
y_train, y_test = y[train_idx], y[test_idx]
scaler = StandardScaler()
x_train[:, :1] = scaler.fit_transform(x_train[:, :1])
x_test[:, :1] = scaler.transform(x_test[:, :1])
if not np.isfinite(x_train).all() or not np.isfinite(x_test).all():
    raise ValueError('Missing or non-finite numeric features')
print('Happiness coefficients below are per training-set standard deviation.')

print("\nMethod 1: Single Sigmoid Neuron")
print("-" * 40)

# Sigmoid activation function
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

# Initializing weights for all features and bias
num_features = X_combined.shape[1]
weights = np.random.randn(num_features) * 0.1
bias = np.random.randn() * 0.1

epochs = 10000
loss_history = []

print("Initial weights:")
print(f"Happiness Score weight: {weights[0]:.6f}")
for i, feature in enumerate(z_features):
    print(f"{feature} weight: {weights[i+1]:.6f}")
print(f"Bias: {bias:.6f}")

# Training loop
for epoch in range(epochs):
    if epoch == 0:
        weights_initial = weights.copy()
        bias_initial = bias
    if epoch == 100:
        print(f"At 100 epochs:")
        print(f"Happiness weight changed from {weights_initial[0]:.8f} to {weights[0]:.8f} (diff: {weights[0]-weights_initial[0]:.8f})")
        print(f"Bias changed from {bias_initial:.8f} to {bias:.8f} (diff: {bias-bias_initial:.8f})")

    # Forward Pass
    z = np.dot(x_train, weights) + bias
    y_pred_prob = sigmoid(z.reshape(-1, 1))

    # Stable binary cross-entropy, matching logistic regression's objective.
    loss = np.mean(np.logaddexp(0, z.reshape(-1, 1)) - y_train * z.reshape(-1, 1))
    loss_history.append(loss)
    current_lr = 0.1
    # BCE + sigmoid: dL/dz = (prediction - target)/n. Average ONCE.
    error = y_pred_prob - y_train
    dL_dweights = (x_train.T @ error).ravel() / len(y_train)
    dL_dbias = error.mean()
    # Gradient Descent (updating the weights)
    weights -= current_lr * dL_dweights
    bias -= current_lr * dL_dbias

    if epoch % 1000 == 0:
        print(f"Epoch {epoch}: Loss: {loss:.6f}, Learning Rate: {current_lr}")

# Final weights for neural network
print("\nFinal weights (Single Sigmoid Neuron):")
print(f"Happiness Score weight: {weights[0]:.6f}")
for i, feature in enumerate(z_features):
    print(f"{feature} weight: {weights[i+1]:.6f}")
print(f"Bias: {bias:.6f}")

# Create equation
equation_parts = [f"{weights[0]:.3f}*happiness"]
for i, feature in enumerate(z_features):
    equation_parts.append(f"{weights[i+1]:.3f}*{feature}")
equation_parts.append(f"{bias:.3f}")
equation = " + ".join(equation_parts)
print(f"Logistic equation: P(y=1) = sigmoid({equation})")

# Testing
z_test = np.dot(x_test, weights) + bias
y_test_pred_prob = sigmoid(z_test.reshape(-1, 1))
y_test_pred_binary = (y_test_pred_prob > 0.5).astype(int)

# Calculating the loss
test_loss = np.mean((y_test - y_test_pred_prob) ** 2)
test_accuracy = np.mean(y_test == y_test_pred_binary)

print(f"\nSingle Sigmoid Neuron Results:")
print(f"Test Loss (MSE): {test_loss:.6f}")
print(f"Test Accuracy: {test_accuracy:.6f}")

print("\nMethod 2: Logistic Regression")
print("-" * 40)

# Creating logistic regression object
logr_norm = linear_model.LogisticRegression(C=1e10, max_iter=5000, tol=1e-10)
logr_norm.fit(x_train, y_train.ravel())

# Get the parameters
weights_norm = logr_norm.coef_[0]
bias_norm = logr_norm.intercept_[0]

print("Sklearn Logistic Regression Parameters:")
print(f"Happiness Score weight: {weights_norm[0]:.6f}")
for i, feature in enumerate(z_features):
    print(f"{feature} weight: {weights_norm[i+1]:.6f}")
print(f"Bias: {bias_norm:.6f}")

# Create equation for normal logistic regression
equation_parts_norm = [f"{weights_norm[0]:.3f}*happiness"]
for i, feature in enumerate(z_features):
    equation_parts_norm.append(f"{weights_norm[i+1]:.3f}*{feature}")
equation_parts_norm.append(f"{bias_norm:.3f}")
equation_norm = " + ".join(equation_parts_norm)
print(f"Logistic equation: P(y=1) = sigmoid({equation_norm})")

# Testing
y_test_pred_prob_norm = logr_norm.predict_proba(x_test)[:, 1]
y_test_pred_binary_norm = logr_norm.predict(x_test)

# Calculating the loss
test_loss_norm = np.mean((y_test.ravel() - y_test_pred_prob_norm) ** 2)
test_accuracy_norm = np.mean(y_test.ravel() == y_test_pred_binary_norm)

print(f"\nSklearn Logistic Regression Results:")
print(f"Test Loss (MSE): {test_loss_norm:.6f}")
print(f"Test Accuracy: {test_accuracy_norm:.6f}")

print("\n" + "=" * 60)
print("Comparison")
print("=" * 60)

print(f"\nParameter Comparison:")
print(f"{'Metric':<25} {'Single Sigmoid Neuron':<15} {'Normal LogReg':<15} {'Difference':<15}")
print("-" * 70)
print(f"{'Happiness Weight':<25} {weights[0]:<15.6f} {weights_norm[0]:<15.6f} {abs(weights[0]-weights_norm[0]):<15.6f}")
for i, feature in enumerate(z_features):
    print(f"{feature:<25} {weights[i+1]:<15.6f} {weights_norm[i+1]:<15.6f} {abs(weights[i+1]-weights_norm[i+1]):<15.6f}")
print(f"{'Bias':<25} {bias:<15.6f} {bias_norm:<15.6f} {abs(bias-bias_norm):<15.6f}")

print(f"\nAccuracy Comparison:")
print(f"{'Metric':<25} {'Single Sigmoid Neuron':<15} {'Normal LogReg':<15} {'Difference':<15}")
print("-" * 70)
print(f"{'Test Loss (MSE)':<25} {test_loss:<15.6f} {test_loss_norm:<15.6f} {abs(test_loss-test_loss_norm):<15.6f}")
print(f"{'Test Accuracy':<25} {test_accuracy:<15.6f} {test_accuracy_norm:<15.6f} {abs(test_accuracy-test_accuracy_norm):<15.6f}")

# Confusion matrices
cm_nn = confusion_matrix(y_test, y_test_pred_binary)
cm_norm = confusion_matrix(y_test, y_test_pred_binary_norm)

print(f"\nConfusion Matrix Comparison:")
print(f"\nSingle Sigmoid Neuron:")
print(f"                Predicted")
print(f"Actual    0    1")
print(f"   0    {cm_nn[0,0]:3d}  {cm_nn[0,1]:3d}")
print(f"   1    {cm_nn[1,0]:3d}  {cm_nn[1,1]:3d}")

print(f"\nSklearn Logistic Regression:")
print(f"                Predicted")
print(f"Actual    0    1")
print(f"   0    {cm_norm[0,0]:3d}  {cm_norm[0,1]:3d}")
print(f"   1    {cm_norm[1,0]:3d}  {cm_norm[1,1]:3d}")

def calculate_metrics(cm):
    precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
    recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

precision_nn, recall_nn, f1_nn = calculate_metrics(cm_nn)
precision_norm, recall_norm, f1_norm = calculate_metrics(cm_norm)

print(f"\nMore Metrics:")
print(f"{'Metric':<25} {'Single Sigmoid Neuron':<15} {'Normal LogReg':<15} {'Difference':<15}")
print("-" * 70)
print(f"{'Precision':<25} {precision_nn:<15.6f} {precision_norm:<15.6f} {abs(precision_nn-precision_norm):<15.6f}")
print(f"{'Recall':<25} {recall_nn:<15.6f} {recall_norm:<15.6f} {abs(recall_nn-recall_norm):<15.6f}")
print(f"{'F1':<25} {f1_nn:<15.6f} {f1_norm:<15.6f} {abs(f1_nn-f1_norm):<15.6f}")

# Do not interpret these coefficients as causal effects or clinical risk.
print('Coefficients describe fitted dataset associations only.')

# Baseline and a genuine hidden-layer network on exactly the same split.
from sklearn.dummy import DummyClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, log_loss, classification_report
baseline = DummyClassifier(strategy='prior').fit(x_train, y_train.ravel())
network = MLPClassifier(hidden_layer_sizes=(8,), activation='tanh', solver='lbfgs',
                        alpha=10.0, max_iter=3000, random_state=42)
network.fit(x_train, y_train.ravel())
probabilities = {
    'Baseline': baseline.predict_proba(x_test)[:, 1],
    'Fixed single neuron': y_test_pred_prob.ravel(),
    'Sklearn logistic': y_test_pred_prob_norm,
    'Hidden-layer network': network.predict_proba(x_test)[:, 1]
}
summary = []
for name, prob in probabilities.items():
    pred = (prob >= 0.5).astype(int)
    summary.append({'Model': name, 'Accuracy': np.mean(pred == y_test.ravel()),
                    'Balanced accuracy': balanced_accuracy_score(y_test.ravel(), pred),
                    'ROC AUC': roc_auc_score(y_test.ravel(), prob),
                    'Log loss': log_loss(y_test.ravel(), prob)})
    print(name)
    print(confusion_matrix(y_test.ravel(), pred, labels=[0, 1]))
    print(classification_report(y_test.ravel(), pred, labels=[0, 1],
          target_names=['None', 'Recorded condition'], zero_division=0))
print(pd.DataFrame(summary).round(4).to_string(index=False))
print('Always predicting condition present gives about 80% accuracy here.')
print('AUC near 0.5 and balanced accuracy near 0.5 indicate little discrimination.')
print('This model predicts dataset labels; it is not a validated diagnostic tool.')
print('Max probability difference from sklearn:',
      np.max(np.abs(y_test_pred_prob.ravel() - y_test_pred_prob_norm)))

# Plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[1,0].plot(loss_history)
axes[1,0].set(xlabel='Epoch', ylabel='Binary cross-entropy', title='Corrected training loss')
# Probability distributions
axes[0,0].hist(y_test.flatten(), bins=30, alpha=0.7, label='Y_actual', color='blue', density=True)
axes[0,0].hist(y_test_pred_prob.flatten(), bins=30, alpha=0.7, label='Y_prediction_probability', color='red', density=True)
axes[0,0].set_xlim(-0.1, 1.1)
axes[0,0].set_xlabel('y values')
axes[0,0].set_ylabel('Density')
axes[0,0].set_title('Single Sigmoid Neuron: Probabilities vs Actual Binary')
axes[0,0].legend()

axes[0,1].hist(y_test.flatten(), bins=30, alpha=0.7, label='Y_actual', color='blue', density=True)
axes[0,1].hist(y_test_pred_prob_norm, bins=30, alpha=0.7, label='Y_prediction_probability', color='green', density=True)
axes[0,1].set_xlim(-0.1, 1.1)
axes[0,1].set_xlabel('y values')
axes[0,1].set_ylabel('Density')
axes[0,1].set_title('Normal LogReg: Probabilities vs Actual Binary')
axes[0,1].legend()

# Mental health rates by category
categories = list(unique_genders) + list(unique_diets)
condition_rates = []

for gender in unique_genders:
    rate = df[df['Gender'] == gender]['condition_binary'].mean()
    condition_rates.append(rate)

for diet in unique_diets:
    rate = df[df['Diet Type'] == diet]['condition_binary'].mean()
    condition_rates.append(rate)

colors = ['lightblue', 'lightcoral', 'lightgreen'] + ['wheat', 'lightyellow', 'lightpink', 'lavender', 'lightgray']
bars = axes[1,1].bar(categories, condition_rates, color=colors[:len(categories)], alpha=0.8)
axes[1,1].set_ylabel('Mental Health Condition Rate')
axes[1,1].set_title('Condition Rates by Demographics')
axes[1,1].set_xticks(range(len(categories)))
axes[1,1].set_xticklabels(categories, rotation=45, ha='right')
axes[1,1].grid(True, alpha=0.3, axis='y')

for bar, rate in zip(bars, condition_rates):
    height = bar.get_height()
    axes[1,1].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{rate:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()